In [1]:
from langchain_community.document_loaders import TextLoader, DataFrameLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma

c:\Users\goyal\OneDrive\Desktop\Book-Recommendation-System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print("All imports Pretty  Successfuly !")

All imports Pretty  Successfuly !


In [3]:
from sentence_transformers import SentenceTransformer

# This will download the model (approx. 80MB) on the first run
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = ["This is an example query", "Each sentence is converted into a vector"]
embeddings = model.encode(sentences)

print(f"Embedding shape: {embeddings.shape}") # Should be (2, 384)

Embedding shape: (2, 384)


In [1]:
import pandas as pd 

books=pd.read_csv("books_science_and_math.csv")

books["tagged_description"].to_csv("new_tagged_descriptions.text",
                                    sep="\n" , index=False, header=False)

In [2]:
# 1. Ensure TextLoader is imported (Fixes the NameError from before)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_data = TextLoader("new_tagged_descriptions.text", encoding="utf-8").load()

# 2. Set chunk_size to 1000 (roughly the length of a book summary)
# and chunk_overlap to 100 (keeps context between chunks)
text_splitter = CharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150, 
    separator="\n"
)

# 3. This will now run without the ValueError
documents = text_splitter.split_documents(raw_data)



c:\Users\goyal\OneDrive\Desktop\Book-Recommendation-System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Created a chunk of size 2764, which is longer than the specified 1500
Created a chunk of size 1773, which is longer than the specified 1500
Created a chunk of size 2007, which is longer than the specified 1500
Created a chunk of size 1599, which is longer than the specified 1500
Created a chunk of size 2211, which is longer than the specified 1500
Created a chunk of size 1640, which is longer than the specified 1500
Created a chunk of size 2122, which is longer than the specified 1500
Created a chunk of size 1693, which is longer than the specified 1500
Created a chunk of size 1634, which is longer than the specified 1500
Created a chunk of size 1893, which is longer than the specified 1500
Created a chu

In [3]:
import os
import shutil
import time
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. SETUP MODEL (This is your all-MiniLM-L6-v2)
model_name = "sentence-transformers/all-MiniLM-L6-v2"
hf_embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={'device': 'cpu'})

# 2. PATH SETUP
db_path = "./book_db_first_version" 
if os.path.exists(db_path):
    shutil.rmtree(db_path)

# 3. INITIAL BATCH (Creates the database)
print("🚀 Starting ingestion...")
batch_size = 40
db_books = Chroma.from_documents(
    documents=documents[:batch_size], 
    embedding=hf_embeddings, 
    persist_directory=db_path
)
print(f"✅ Initial {batch_size} books stored.")

# 4. LOOP FOR REMAINING BOOKS
for i in range(batch_size, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    db_books.add_documents(batch)
    print(f"✅ Progress: {i + len(batch)} / {len(documents)} books stored.")
    time.sleep(5) # Small pause to let the CPU breathe

print("\n✨ ALL BOOKS STORED SUCCESSFULLY!")

🚀 Starting ingestion...
✅ Initial 40 books stored.
✅ Progress: 80 / 121 books stored.
✅ Progress: 120 / 121 books stored.
✅ Progress: 121 / 121 books stored.

✨ ALL BOOKS STORED SUCCESSFULLY!


In [ ]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Re-initialize the exact same model wrapper used for building
model_name = "sentence-transformers/all-MiniLM-L6-v2"
hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)

# 2. Path to your saved folder
db_path = "./book_db_final_version"

if os.path.exists(db_path):
    # This LOADS the database from disk
    db_books = Chroma(
        persist_directory=db_path,
        embedding_function=hf_embeddings  # Must use the wrapper here!
    )
    
    # Verify the count
    num_books = len(db_books.get()['ids'])
    print(f"✅ Database loaded successfully with {num_books} books.")
else:
    print(f"❌ Error: Folder '{db_path}' not found. Run the ingestion code first!")

✅ Database loaded successfully with 2139 books.


In [ ]:
query = "A book For Math"
results = db_books.similarity_search(query, k=10)
print("results: ", results)

results:  [Document(id='2d4203c8-9623-4f2f-85f3-45d4bee328c7', metadata={'source': 'new_tagged_descriptions.text'}, page_content='9780691050843 | Presents a series of lectures delivered in 1994 by Hawking and Penrose, renowned professors at Cambridge and Oxford, respectively, on the general topic of how mathematical physics might best represent the realities of the universe.'), Document(id='37c1d2f2-c389-4c9a-9942-c766f2ff567c', metadata={'source': 'new_tagged_descriptions.text'}, page_content='9780195325928 | Mathematics and logic have been central topics of concern since the dawn of philosophy. Since logic is the study of correct reasoning, it is a fundamental branch of epistemology and a priority in any philosophical system. Philosophers have focused on mathematics as a case study for general philosophical issues and for its role in overall knowledge- gathering. Today, philosophy of mathematics and logic remain central disciplines in contemporary philosophy, as evidenced by the regu

In [ ]:
# .replace('"', '') removes any double quotes found in that first segment
isbn_str = results[0].page_content.split(" ")[0].replace('"', '').strip()
books[books["isbn13"] == int(isbn_str)]
# print(results[0].metadata)

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
137,9780691050843,0691050848,The Nature of Space and Time,Stephen Hawking;Roger Penrose,Science,http://books.google.com/books/content?id=LstaQ...,2000.0,4.09,152.0,945.0,The Nature of Space and Time,9780691050843 | Presents a series of lectures ...


In [ ]:
def retrieve_symmentaic_recommendations(
        query: str,
        top_k: int = 10
    ,
)-> pd.DataFrame:
    recs=db_books.similarity_search(query, k=top_k)
    book_list=[]
    for i in range(len(recs)):
        isbn_str = recs[i].page_content.split(" ")[0].replace('"', '').strip()
        book_info=books[books["isbn13"] == int(isbn_str)]
        book_list.append(book_info)
    return pd.concat(book_list).reset_index(drop=True)
    

In [10]:
results_df=retrieve_symmentaic_recommendations("A book about a Time and Space",top_k=10)
results_df

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780141011110,0141011114,The Fabric of the Cosmos,Brian Greene,Science,http://books.google.com/books/content?id=dpSqv...,2005.0,4.12,592.0,324.0,"The Fabric of the Cosmos: Space, Time and the ...",9780141011110 | From the bestselling author of...
1,9780140286014,0140286012,Wanderlust,Rebecca Solnit,Social Science,http://books.google.com/books/content?id=_R2MD...,2001.0,3.92,326.0,209.0,Wanderlust: A History of Walking,9780140286014 | A cultural history of walking ...
2,9780553380163,0553380168,A Brief History of Time,Stephen Hawking,Science,http://books.google.com/books/content?id=YmGxb...,1998.0,4.16,212.0,214520.0,A Brief History of Time,9780553380163 | An anniversary edition of a no...
3,9780679776314,0679776311,The Road to Reality,Roger Penrose,Science,http://books.google.com/books/content?id=coahA...,2007.0,4.10,1099.0,5892.0,The Road to Reality: A Complete Guide to the L...,9780679776314 | Presents an overview of the ph...
4,9780393324464,039332446X,The Future of Spacetime,Stephen W. Hawking;Kip S. Thorne;Igor D. Novik...,Science,http://books.google.com/books/content?id=LlVcB...,2003.0,3.93,224.0,241.0,The Future of Spacetime,9780393324464 | Presents essays that explore t...
5,9780691050843,0691050848,The Nature of Space and Time,Stephen Hawking;Roger Penrose,Science,http://books.google.com/books/content?id=LstaQ...,2000.0,4.09,152.0,945.0,The Nature of Space and Time,9780691050843 | Presents a series of lectures ...
6,9780802713520,0802713521,E=mc2,David Bodanis,Science,http://books.google.com/books/content?id=Y5FwQ...,2000.0,4.09,352.0,157.0,E=mc2: A Biography of the World's Most Famous ...,9780802713520 | Generations have grown up know...
7,9780761956921,0761956921,Société de Consommation,Jean Baudrillard,Social Science,http://books.google.com/books/content?id=Bbex0...,1998.0,4.13,224.0,654.0,"Société de Consommation: Ses Mythes, Ses Struc...",9780761956921 | Now available in English for t...
8,9780465070596,0465070590,Shattered Bonds,Dorothy E. Roberts,Political Science,http://books.google.com/books/content?id=D47-K...,2002.0,4.20,352.0,145.0,Shattered Bonds: The Color of Child Welfare,9780465070596 | Identifies a disproportionate ...
9,9780965738408,096573840X,A Short History of Nearly Everything,Bill Bryson,Science,NaN,2003.0,4.20,545.0,28.0,A Short History of Nearly Everything: 7 Steps ...,9780965738408 | In this book Bill Bryson explo...


In [11]:
books['categories'].value_counts().reset_index().query('count>50')

,categories,count
0,Science,56


In [ ]:
category_mapping = {
    'Fiction': "Fiction",
    'Juvenile Fiction': "Fiction",
    'Comics & Graphic Novels': "Fiction",
    'Drama': "Fiction",
    'Poetry': "Fiction",
    'Biography & Autobiography': "NonFiction",
    'History': "NonFiction",
    'Literary Criticism': "NonFiction",
    'Philosophy': "NonFiction",
    'Religion': "NonFiction",
    'Science': "NonFiction",
    'Juvenile Nonfiction': "NonFiction"
}
books['broad_category'] = books['categories'].map(category_mapping)

In [10]:
books[~(books['broad_category'].isna())]
# books['broad_category'].value_counts().reset_index().query('count>50')

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,broad_category
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 | A NOVEL THAT READERS and criti...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 | A memorable, mesmerizing heroi...",Fiction
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0f...,1995.0,4.03,522.0,2966.0,Warhost of Vastmark,9780006482079 | Tricked once more by his wily ...,Fiction
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,2002.0,3.50,32.0,1.0,Ocean Star Express,9780006646006 | Joe and his parents are enjoyi...,Fiction
46,9780007121014,0007121016,Taken at the Flood,Agatha Christie,Fiction,http://books.google.com/books/content?id=3gWlx...,2002.0,3.71,352.0,8852.0,Taken at the Flood,9780007121014 | A Few Weeks After Marrying An ...,Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5178,9781933648279,1933648279,Night Has a Thousand Eyes,Cornell Woolrich,Fiction,http://books.google.com/books/content?id=3Gk6s...,2007.0,3.77,344.0,680.0,Night Has a Thousand Eyes,"9781933648279 | ""Cornell Woolrich's novels def...",Fiction
5188,9784770028969,4770028962,Coin Locker Babies,村上龍,Fiction,http://books.google.com/books/content?id=87DJw...,2002.0,3.75,393.0,5560.0,Coin Locker Babies,9784770028969 | Rescued from the lockers in wh...,Fiction
5189,9788122200850,8122200850,"Cry, the Peacock",Anita Desai,Fiction,http://books.google.com/books/content?id=_QKwV...,1980.0,3.22,218.0,134.0,"Cry, the Peacock",9788122200850 | This book is the story of a yo...,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 | This collection of the timeles...,NonFiction


In [11]:
from transformers import pipeline,AutoConfig

# Initialize the lightweight classifier
# model="valhalla/distilbart-mnli-12-3" is the faster version
config = AutoConfig.from_pretrained("valhalla/distilbart-mnli-12-3")
config.tie_word_embeddings = False
fiction_categories = ["Fiction", "NonFiction"]
classifier = pipeline(
    "zero-shot-classification", 
    model="valhalla/distilbart-mnli-12-3",
    device=-1  # This forces the model to run on your CPU
)


Device set to use cpu


In [12]:
sequence= books.loc[books['broad_category'] == "Fiction", "tagged_description"].reset_index(drop=True)[0]

In [13]:
# # from os import pipe
# result=classifier(sequence,fiction_categories)
clean_sequence = sequence.split('|')[-1].strip()

# # 3. Classify the cleaned text
result = classifier(clean_sequence, fiction_categories)

print(f"Label: {result['labels'][0]} with score: {result['scores'][0]:.4f}")

Label: Fiction with score: 0.7222


In [14]:
import numpy as np
max_index=np.argmax(classifier(sequence,fiction_categories)['scores'])
my_label=classifier(sequence,fiction_categories)['labels'][max_index]
my_label

'Fiction'

In [ ]:
def generate_predictions(sequence, function_categories):
    prediction = classifier(sequence, function_categories)
    max_index = np.argmax(prediction['scores'])
    max_label = prediction['labels'][max_index]
    return max_label

In [16]:
from  tqdm import tqdm
actual_cate = []
predicted_cate = []

# Filter once to avoid the indexing bug
fiction_subset = books[books["broad_category"] == "Fiction"].reset_index(drop=True)

for i in tqdm(range(0, 300)):
    sequence = fiction_subset["tagged_description"][i]
    
    pred = generate_predictions(sequence, fiction_categories)
    
    predicted_cate.append(pred)
    actual_cate.append("Fiction")

100%|██████████| 300/300 [04:28<00:00,  1.12it/s]


In [ ]:
# Filter once to avoid the indexing bug
fiction_subset = books[books["broad_category"] == "NonFiction"].reset_index(drop=True)

for i in tqdm(range(0, 300)):
    sequence = fiction_subset["tagged_description"][i]
    
    pred = generate_predictions(sequence, fiction_categories)
    
    predicted_cate.append(pred)
    actual_cate.append("NonFiction")

100%|██████████| 300/300 [04:44<00:00,  1.05it/s]


In [ ]:
predictions_df=pd.DataFrame({"actual_category": actual_cate, "predicted_category": predicted_cate})

In [19]:
predictions_df

,actual_category,predicted_category
0,Fiction,Fiction
1,Fiction,Fiction
2,Fiction,Fiction
3,Fiction,Fiction
4,Fiction,Fiction
...,...,...
595,NonFiction,NonFiction
596,NonFiction,NonFiction
597,NonFiction,NonFiction
598,NonFiction,NonFiction


In [20]:
predictions_df["correct_prediction"]=np.where(predictions_df["actual_category"]==predictions_df["predicted_category"], 1, 0)    

In [21]:
predictions_df["correct_prediction"].sum()/len(predictions_df)

np.float64(0.825)

In [22]:
isbns=[]
predicted_cats=[]

missing_cats=books.loc[books["broad_category"].isna(), ["isbn13", "tagged_description"]].reset_index(drop=True)

In [23]:
for i in tqdm(range(len(missing_cats))):
    sequence = missing_cats["tagged_description"][i]
    
    pred = generate_predictions(sequence, fiction_categories)
    
    predicted_cats.append(pred)
    isbns.append(missing_cats["isbn13"][i]) 

100%|██████████| 1454/1454 [21:26<00:00,  1.13it/s]


In [24]:
missing_predictions_df=pd.DataFrame({"isbn13": isbns, "predicted_category": predicted_cats})
missing_predictions_df

,isbn13,predicted_category
0,9780002261982,Fiction
1,9780006280897,NonFiction
2,9780006280934,NonFiction
3,9780006380832,NonFiction
4,9780006470229,Fiction
...,...,...
1449,9788125026600,NonFiction
1450,9788171565641,Fiction
1451,9788172235222,Fiction
1452,9788173031014,NonFiction


In [25]:
books=pd.merge(books, missing_predictions_df, on="isbn13", how="left")
books["books_category"]=np.where(books["broad_category"].isna(), books["predicted_category"], books["broad_category"])
books=books.drop(columns=["predicted_category"])

In [26]:
books.to_csv("books_with_categories.csv", index=False)